# Tổng hợp năm phương án đặc trưng

Chạy notebook này sau V1–V5 để so sánh MAE/RMSE CV và chọn cấu hình cho tuning. Notebook đọc các kết quả đã lưu, kiểm tra tính đồng nhất rồi tổng hợp; không huấn luyện thêm mô hình.

In [1]:
from pathlib import Path
import sys
project_root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
                    if (folder / 'data/Metro_Interstate_Traffic_Volume.csv').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import pandas as pd
from IPython.display import Markdown, display
from src import feature_experiments as fe

## 1. Kiểm tra protocol

Cả năm phải cùng phiên bản dữ liệu, preprocessing, RF và tham số, cohort và fold. Các dấu hash giúp phát hiện thay đổi dữ liệu hoặc thứ tự dòng. Chỉ dùng CV trong train; chưa chấm validation/test.

In [2]:
context = fe.load_protocol(project_root)
display(context['manifest'])
display(Markdown(f"Protocol `{context['protocol_id']}`; cùng {len(context['cohort']):,} giờ trong cohort. Các fold bằng số dòng chấm, có thể khác thời lượng lịch."))

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...


Protocol `e4bc1e064317af8c`; cùng 22,503 giờ trong cohort. Các fold bằng số dòng chấm, có thể khác thời lượng lịch.

## 2. Bảng so sánh năm phương án

Chọn MAE trung bình fold thấp nhất; dùng RMSE để đối chiếu và phá hòa. Độ lệch chuẩn mô tả biến động giữa các giai đoạn, không phải khoảng tin cậy. Nếu thiếu run cùng protocol, bảng hiện “chưa chạy” và chưa chốt. Khi đủ năm kết quả, ô sau cập nhật bảng tổng hợp, `feature_summary.md` và xuất dữ liệu chưa fit preprocessing cho cấu hình được chọn.

In [3]:
comparison = fe.compare_configurations(context)
display(comparison)

,rank,config,n_features,status,MAE_mean,MAE_std,RMSE_mean,RMSE_std,n
0,1,V5_C30,30,đã chạy,247.926016,12.404353,412.906913,44.346083,16875
1,2,V1_A12,12,đã chạy,288.318165,35.451861,469.685869,56.277505,16875
2,3,V3_labels23,23,đã chạy,290.037162,34.118225,470.751748,59.287338,16875
3,4,V4_B27,27,đã chạy,290.725342,29.323710,473.646347,59.538782,16875
4,5,V2_numeric16,16,đã chạy,291.552831,29.059430,476.418740,56.677147,16875


## 3. Mức thay đổi so với A12

Dùng A12 làm mốc để xem mỗi cấu hình thay đổi sai số bao nhiêu xe/giờ và giảm MAE ở bao nhiêu fold. Dấu âm nghĩa là sai số giảm. Đây là đối chứng nhóm đặc trưng, chưa tách đóng góp từng cột.

In [4]:
if comparison.status.eq('đã chạy').all():
    baseline = pd.read_csv(context['output'] / 'V1_A12/cv_scores.csv').set_index('fold')
    changes = []
    for config in fe.CONFIGS:
        current = pd.read_csv(context['output'] / config / 'cv_scores.csv').set_index('fold')
        changes.append(dict(config=config, delta_MAE=(current.MAE - baseline.MAE).mean(),
                            delta_RMSE=(current.RMSE - baseline.RMSE).mean(),
                            folds_lower_MAE=int((current.MAE < baseline.MAE).sum()),
                            n=int(current.n.sum())))
    display(pd.DataFrame(changes))
else:
    display(Markdown('Cần đủ năm run trước khi tổng hợp chênh lệch.'))

,config,delta_MAE,delta_RMSE,folds_lower_MAE,n
0,V1_A12,0.000000,0.000000,0,16875
1,V2_numeric16,3.234665,6.732870,1,16875
2,V3_labels23,1.718996,1.065879,1,16875
3,V4_B27,2.407177,3.960477,1,16875
4,V5_C30,-40.392149,-56.778956,3,16875


## 4. Quyết định và bàn giao cho tuning

In [5]:
if comparison.status.eq('đã chạy').all():
    winner = comparison.iloc[0]
    display(Markdown(f"Chọn **{winner['config']}** theo MAE CV {winner['MAE_mean']:.3f} xe/giờ; RMSE CV {winner['RMSE_mean']:.3f} xe/giờ. Dữ liệu bàn giao nằm trong `selected/{context['protocol_id']}/`, gồm feature chưa điền thiếu, ngữ cảnh theo giờ và `selection.json`."))
else:
    display(Markdown('Chưa chọn cấu hình. Chạy các notebook còn thiếu rồi chạy lại notebook tổng hợp này.'))

Chọn **V5_C30** theo MAE CV 247.926 xe/giờ; RMSE CV 412.907 xe/giờ. Dữ liệu bàn giao nằm trong `selected/e4bc1e064317af8c/`, gồm feature chưa điền thiếu, ngữ cảnh theo giờ và `selection.json`.

Lựa chọn này áp dụng cho RF cố định và cohort chung. Khi tuning, giữ chia thời gian và fit lại preprocessing trong từng fold; chưa dùng test để chọn tham số. History giả định quan sát quá khứ đến kịp; chưa đại diện dự báo nhiều bước. Weather cùng giờ được giả định sẵn có.